# **Day 79: Making Code Faster - Measure First**

A famous rule in programming: **"First make it work, then make it fast."**

And before making anything faster, we must **measure** it. Otherwise we might waste time on a part that was never slow.

## **timeit: Measuring Small Pieces of Code**
In Jupyter, `%timeit` runs the code many times and shows the average time.

In [1]:
%timeit sum(range(10_000))

62.1 μs ± 873 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [2]:
%timeit sum([i for i in range(10_000)])

191 μs ± 571 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## **Common Speed Tips (Measured)**
### **1. Use a Set for Searching**

In [3]:
pins_list = list(range(700000, 750000))
pins_set = set(pins_list)
%timeit 749999 in pins_list
%timeit 749999 in pins_set

248 μs ± 1.42 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


21.1 ns ± 0.117 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


Searching in a set is **thousands of times faster**, because a set jumps directly to the item, while a list checks one by one.

### **2. Use join() for Building Strings**

In [4]:
def with_plus(n):
    s = ""
    for i in range(n):
        s += str(i)
    return s

def with_join(n):
    return "".join(str(i) for i in range(n))

%timeit with_plus(10_000)
%timeit with_join(10_000)

1.35 ms ± 121 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


567 μs ± 4.17 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### **3. List Comprehension vs Loop**

In [5]:
nums = list(range(100_000))

def with_loop():
    out = []
    for n in nums:
        out.append(n * n)
    return out

%timeit with_loop()
%timeit [n * n for n in nums]

2.85 ms ± 50 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


2.34 ms ± 29.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### **4. The Biggest Speed-Up: A Better Method (Algorithm)**
Checking for duplicates by comparing every pair takes a very long time for big lists. Using a set is much faster.

In [6]:
def has_duplicate_slow(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False

def has_duplicate_fast(items):
    return len(set(items)) != len(items)

data = list(range(3000))
%timeit has_duplicate_slow(data)
%timeit has_duplicate_fast(data)

105 ms ± 524 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


24.1 μs ± 407 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## **Big-O: How Time Grows with Data Size**
**Big-O** describes how much slower a program gets when the data becomes bigger.

| Big-O | Name | Simple meaning | Example |
|---|---|---|---|
| O(1) | Constant | Same time for any size | Looking up a dictionary key |
| O(log n) | Logarithmic | Grows very slowly | Binary search |
| O(n) | Linear | Double the data, double the time | One loop |
| O(n log n) | | A bit more than linear | Sorting |
| O(n²) | Quadratic | Double the data, **four times** the time | A loop inside a loop |

The slow duplicate check is O(n²), and the set method is O(n).

## **Example from Remote Sensing: Python Loops vs NumPy**
A single Sentinel-2 band has about **120 million pixels**. Calculating NDVI with a normal Python loop is very slow. The **NumPy** library (Day 81) does the same maths on the whole image at once, using fast code written in C.

Let us compare on 1 million pixels.

In [7]:
import random
import numpy as np

random.seed(0)
n = 1_000_000
red_list = [random.uniform(0.02, 0.2) for _ in range(n)]
nir_list = [random.uniform(0.05, 0.5) for _ in range(n)]
red_array = np.array(red_list)
nir_array = np.array(nir_list)

def ndvi_python():
    return [(nr - r) / (nr + r) for r, nr in zip(red_list, nir_list)]

def ndvi_numpy():
    return (nir_array - red_array) / (nir_array + red_array)

%timeit ndvi_python()
%timeit ndvi_numpy()

57.5 ms ± 450 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


5.44 ms ± 54.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


NumPy is **many times faster**. This is why all serious Remote Sensing work in Python uses NumPy.

## **Profiling: Finding the Slow Part**
A **profiler** tells us **which function** is taking the most time.

In [8]:
import cProfile, pstats, io
from collections import defaultdict

def load_data():
    random.seed(0)
    return [(random.choice(["Kolkata", "Howrah", "Siliguri", "Durgapur"]), random.randint(100, 5000))
            for _ in range(60_000)]

def slow_group(rows):
    cities = []
    for c, _ in rows:
        if c not in cities:
            cities.append(c)
    return {c: sum(a for cc, a in rows if cc == c) for c in cities}     # Reads all rows again for each city

def report():
    return slow_group(load_data())

profiler = cProfile.Profile()
profiler.enable()
report()
profiler.disable()

output = io.StringIO()
pstats.Stats(profiler, stream=output).sort_stats("cumulative").print_stats(6)
print("\n".join(line for line in output.getvalue().splitlines() if "{" not in line and line.strip())[:1500])

         879993 function calls in 0.216 seconds
   Ordered by: cumulative time
   List reduced from 62 to 6 due to restriction <6>
   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.186    0.093 C:\Users\Manas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3786(run_code)
        1    0.045    0.045    0.186    0.186 C:\Users\Manas\AppData\Local\Temp\ipykernel_124240\2623298309.py:1(<module>)
        1    0.001    0.001    0.141    0.141 C:\Users\Manas\AppData\Local\Temp\ipykernel_124240\2623298309.py:16(report)
        1    0.017    0.017    0.124    0.124 C:\Users\Manas\AppData\Local\Temp\ipykernel_124240\2623298309.py:4(load_data)
   120000    0.045    0.000    0.073    0.000 C:\Users\Manas\AppData\Local\Python\pythoncore-3.14-64\Lib\random.py:245(_randbelow_with_getrandbits)


The profile shows where the time goes. A better grouping reads the data **only once**:

In [9]:
def fast_group(rows):
    totals = defaultdict(int)
    for c, a in rows:
        totals[c] += a
    return dict(totals)

rows = load_data()
print(fast_group(rows) == slow_group(rows))       # Same answer
%timeit slow_group(rows)
%timeit fast_group(rows)

True


9.32 ms ± 101 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


2.64 ms ± 24.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## **Checklist for Faster Code**
1. Is it really slow? **Measure** it.
2. **Profile** to find the slow part.
3. Use a **better method** or data structure (set, dictionary).
4. Use **built-in functions** and libraries like **NumPy**.
5. **Cache** results that are calculated again and again (`lru_cache`).
6. Only then, try multiprocessing.

In [10]:
# Quick question
# A loop inside a loop over n items is O(?)